In [2]:
pip install gradio PyMuPDF pillow

   ---------------------------------------- 0.0/18.7 MB ? eta -:--:--
   -------------------- ------------------- 9.7/18.7 MB 51.5 MB/s eta 0:00:01
   ------------------------------- -------- 14.7/18.7 MB 37.9 MB/s eta 0:00:01
   ------------------------------------ --- 17.3/18.7 MB 29.1 MB/s eta 0:00:01
   ---------------------------------------  18.6/18.7 MB 26.9 MB/s eta 0:00:01
   ---------------------------------------- 18.7/18.7 MB 22.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import gradio as gr
import fitz  # PyMuPDF
from pathlib import Path
from PIL import Image

def convert_pdfs_to_jpg(folder_path, output_folder=None, dpi=150):
    """
    Convert all PDF files in a folder to JPG images.
    
    Args:
        folder_path: Path to folder containing PDF files
        output_folder: Path to output folder (optional, defaults to folder_path/output)
        dpi: Resolution for output images (default: 150)
    
    Returns:
        Status message with conversion details
    """
    try:
        # Validate folder path
        if not folder_path or not os.path.exists(folder_path):
            return "❌ Error: Invalid folder path. Please provide a valid directory path."
        
        folder_path = Path(folder_path)
        
        # Set output folder
        if output_folder:
            output_path = Path(output_folder)
        else:
            output_path = folder_path / "output"
        
        # Create output folder if it doesn't exist
        output_path.mkdir(parents=True, exist_ok=True)
        
        # Find all PDF files
        pdf_files = list(folder_path.glob("*.pdf")) + list(folder_path.glob("*.PDF"))
        
        if not pdf_files:
            return f"❌ No PDF files found in {folder_path}"
        
        total_pages = 0
        converted_files = []
        
        # Process each PDF
        for pdf_file in pdf_files:
            try:
                # Open PDF
                pdf_document = fitz.open(pdf_file)
                pdf_name = pdf_file.stem
                
                # Convert each page
                for page_num in range(len(pdf_document)):
                    page = pdf_document[page_num]
                    
                    # Set zoom for desired DPI (default 72 DPI in PyMuPDF)
                    zoom = dpi / 72
                    mat = fitz.Matrix(zoom, zoom)
                    
                    # Render page to pixmap
                    pix = page.get_pixmap(matrix=mat)
                    
                    # Create output filename
                    output_filename = f"{pdf_name}_page_{page_num + 1}.jpg"
                    output_file = output_path / output_filename
                    
                    # Save as JPG
                    pix.save(output_file)
                    total_pages += 1
                
                pdf_document.close()
                converted_files.append(pdf_name)
                
            except Exception as e:
                return f"❌ Error processing {pdf_file.name}: {str(e)}"
        
        # Success message
        result = f"✅ Successfully converted {len(converted_files)} PDF file(s)\n"
        result += f"📄 Total pages converted: {total_pages}\n"
        result += f"📁 Output folder: {output_path}\n\n"
        result += "Converted files:\n" + "\n".join([f"  • {name}" for name in converted_files])
        
        return result
        
    except Exception as e:
        return f"❌ Error: {str(e)}"

# Create Gradio interface
def create_interface():
    with gr.Blocks(title="PDF to JPG Converter", theme=gr.themes.Soft()) as demo:
        gr.Markdown(
            """
            # 📄 PDF to JPG Converter
            Convert all PDF files in a folder to high-quality JPG images.
            Each page will be saved as a separate image file.
            """
        )
        
        with gr.Row():
            with gr.Column():
                folder_input = gr.Textbox(
                    label="Folder Path",
                    placeholder="Enter the path to folder containing PDF files (e.g., C:/Documents/PDFs)",
                    lines=1
                )
                
                output_folder_input = gr.Textbox(
                    label="Output Folder (Optional)",
                    placeholder="Leave empty to create 'output' subfolder in input folder",
                    lines=1
                )
                
                dpi_slider = gr.Slider(
                    minimum=72,
                    maximum=300,
                    value=150,
                    step=1,
                    label="Image Quality (DPI)",
                    info="Higher DPI = better quality but larger file size"
                )
                
                convert_btn = gr.Button("🔄 Convert PDFs", variant="primary", size="lg")
        
        with gr.Row():
            output_text = gr.Textbox(
                label="Conversion Status",
                lines=10,
                interactive=False
            )
        
        gr.Markdown(
            """
            ### 📝 Instructions:
            1. Enter the full path to the folder containing your PDF files
            2. (Optional) Specify a custom output folder path
            3. Adjust the DPI slider for desired image quality
            4. Click "Convert PDFs" to start the conversion
            
            ### 💡 Tips:
            - Default output folder will be created as 'output' inside your input folder
            - DPI 150 is good for most uses, 300 for high-quality prints
            - Files are named as: `originalname_page_1.jpg`, `originalname_page_2.jpg`, etc.
            """
        )
        
        # Connect button to function
        convert_btn.click(
            fn=convert_pdfs_to_jpg,
            inputs=[folder_input, output_folder_input, dpi_slider],
            outputs=output_text
        )
    
    return demo

# Launch the app
if __name__ == "__main__":
    demo = create_interface()
    demo.launch(share=False)

c:\Users\r.musawi\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
